In [1]:
# Подключаемся к проекту
import sys
import os
from pathlib import Path

# Поднимаемся на уровень выше (из notebooks в корень проекта)
project_root = Path.cwd().parent
sys.path.append(str(project_root))

print(f"📁 Корень проекта: {project_root}")
print(f"📁 База данных: {project_root / 'data' / 'olfactory.db'}")

📁 Корень проекта: c:\Users\lolla\Desktop\zapiski_sumashedshego\dev_null
📁 База данных: c:\Users\lolla\Desktop\zapiski_sumashedshego\dev_null\data\olfactory.db


In [2]:
# Импортируем классы
from src.database import OlfactoryDB
from notebooks.research import OlfactoryResearch

# Создаем объект для исследований
db_path = project_root / "data" / "olfactory.db"
research = OlfactoryResearch(str(db_path))

print("✅ Готово!")

✅ Готово!


In [3]:
df = research.get_texts_dataframe()
df.head()

,id,title,author,year,content,text_type,created_at,language,smell_count,token_count,smell_ratio
0,1,prestuplenie,dostoewski,None,Ф. ДОСТОЕВСКИЙ\n\nПРЕСТУПЛЕНИЕ И НАКАЗАНИЕ\n\n...,original,2026-03-18 06:27:30,ru,17,0,0
1,1,prestuplenie,-,None,FYODOR DOSTOEVSKY\n\nCRIME AND PUNISHMENT\n\n\...,translation,2026-03-18 06:27:35,en,28,0,0


## 1. **Базовые команды для знакомства с данными**

In [ ]:
# Посмотреть все тексты
df_texts = research.get_texts_dataframe()
df_texts.head(20)  # первые 20 строк

# Информация о таблице
df_texts.info()

# Основная статистика
df_texts.describe()

# Сколько текстов по языкам
df_texts['language'].value_counts()

## 2. **Анализ текстов**

In [ ]:
# Тексты с наибольшим количеством запахов
df_texts.nlargest(10, 'smell_count')[['title', 'author', 'language', 'smell_count']]

# Тексты без запахов
df_texts[df_texts['smell_count'] == 0][['title', 'author']]

# Среднее количество запахов по языкам
df_texts.groupby('language')['smell_count'].mean()

## 3. **Исследование самих запахов**

In [ ]:
# Получить все слова-запахи
smells_df = research.get_smells_dataframe()
smells_df.head()

# Сколько всего уникальных запахов
smells_df['word'].nunique()

# Топ-30 самых частых запахов
smells_df['word'].value_counts().head(30)

# Какие части речи чаще всего используются для запахов
smells_df['pos'].value_counts()

## 4. **Поиск и фильтрация**

In [ ]:
# Найти все тексты с конкретным запахом
apple_texts = research.search_by_smell("яблоко", language='ru')
apple_texts[['title', 'author', 'sentence']]

# Найти тексты с несколькими запахами (например, "роза" и "сирень")
rose_texts = research.search_by_smell("роза")
lilac_texts = research.search_by_smell("сирень")

# Статистика по конкретному автору
dostoevsky_stats = research.get_smell_statistics_by_author("Достоевский")

## 5. **Графики и визуализация**

In [ ]:

# Распределение запахов
research.plot_smell_distribution('ru')

# Топ-30 самых частых запахов в русских текстах
research.plot_top_smells('ru', n=30)

# Сравнение русского и английского
research.compare_languages()

# Сравнение всех языков, которые есть
for lang in ['ru', 'en', 'de']:
    print(f"\n--- {lang.upper()} ---")
    research.plot_top_smells(lang, n=15)

## 6. **Сравнение авторов**

In [ ]:
# Сравнить двух авторов
tolstoy = research.get_smell_statistics_by_author("Толстой")
dostoevsky = research.get_smell_statistics_by_author("Достоевский")

if tolstoy is not None and dostoevsky is not None:
    print("Толстой - топ запахов:")
    print(tolstoy['smell_word'].value_counts().head(10))
    print("\nДостоевский - топ запахов:")
    print(dostoevsky['smell_word'].value_counts().head(10))

## 7. **Динамика по годам**

In [ ]:
# Как менялось количество запахов по годам
df = research.get_texts_dataframe()
yearly = df.groupby('year')['smell_count'].mean().dropna()
yearly.plot(kind='line', figsize=(12, 6), title='Среднее количество запахов по годам')
plt.show()

# Какой год самый "пахучий"
df.groupby('year')['smell_count'].sum().nlargest(10)

## 8. **Экспорт данных для внешнего анализа**

In [ ]:
# Экспортировать все русские запахи в CSV
research.export_to_csv('olfactory_ru', 'русские_запахи.csv')

# Экспортировать все тексты
df_texts.to_csv('все_тексты.csv', index=False)

# Экспортировать топ-100 запахов
top100 = smells_df['word'].value_counts().head(100)
top100.to_csv('топ_100_запахов.csv')

## 9. **Корреляции и закономерности**

In [ ]:
import seaborn as sns

# Есть ли связь между годом и количеством запахов?
df = research.get_texts_dataframe()
sns.scatterplot(data=df, x='year', y='smell_count', hue='language')
plt.title('Запахи по годам')
plt.show()

# Какие авторы пишут о запахах чаще других?
author_stats = df.groupby('author')['smell_count'].agg(['mean', 'sum', 'count'])
author_stats.sort_values('mean', ascending=False).head(20)

## 10. **Исследование контекста** (если хотите углубиться)

In [ ]:
# Создайте новый метод в research.py для анализа контекста
def analyze_context(self, smell_word, language=None):
    """Анализирует, в каком контексте встречается запах"""
    # TODO: добавить анализ left_context и right_context
    pass

## Полный пример исследования в одной ячейке:

```python
print("="*60)
print("📊 ИССЛЕДОВАНИЕ ЗАПАХОВ В РУССКОЙ ЛИТЕРАТУРЕ")
print("="*60)

# 1. Общая статистика
df = research.get_texts_dataframe('ru')
print(f"\n📚 Всего текстов: {len(df)}")
print(f"👃 Всего запахов: {df['smell_count'].sum()}")
print(f"📈 В среднем на текст: {df['smell_count'].mean():.2f}")

# 2. Самые "пахучие" тексты
print("\n🏆 ТОП-5 ТЕКСТОВ ПО ЗАПАХАМ:")
top5 = df.nlargest(5, 'smell_count')[['title', 'author', 'year', 'smell_count']]
print(top5.to_string(index=False))

# 3. Самые частые запахи
print("\n🔝 ТОП-20 САМЫХ ЧАСТЫХ ЗАПАХОВ:")
smells = research.get_smells_dataframe('ru')
top20 = smells['word'].value_counts().head(20)
for i, (word, count) in enumerate(top20.items(), 1):
    print(f"  {i:2d}. {word:<15} {count:3d} раз")

# 4. Графики
research.plot_smell_distribution('ru')
research.plot_top_smells('ru', n=25)

# 5. Сохраняем результаты
research.export_to_csv('olfactory_ru', 'результаты_исследования.csv')
print("\n✅ Результаты сохранены в 'результаты_исследования.csv'")
```